# CricketMind — how the graph is built

The architecture of Figure 1, assembled one stage at a time. Each stage is the
smallest graph that demonstrates one LangGraph behaviour, and each ends with a
check that has to pass before the next stage is added.

**Nothing is defined here.** Every node, every graph and every constant lives in
the `cricketmind` package at the repository root, so this notebook cannot drift
from the code the tests run against. To change behaviour, edit the package.

| Stage | Graph | What it shows |
|---|---|---|
| 2 | `build_vision_graph` | a linear chain, N1 → N2 → N4 → N5 |
| 3 | `build_gated_graph` | a conditional edge and a bounded retry loop |
| 4 | `build_grounded_graph` | two branches off START, joining at N6 |
| 5 | `build_retrieval_graph` | a fan-out whose parallel writes need reducers |
| 6 | `build_analytics_graph` | `defer=True` for parents at different depths |
| 7 | `build_router_graph` | dispatch computed from state |
| 8 | `build_agent_graph` | a conditional edge returning a LIST |
| — | `build_cricketmind_graph` | all 30 nodes, both loops, both entry paths |

**Hardware note:** this runs unmodified on both machines — Intel Arc B580 (`xpu`)
and NVIDIA RTX 3070 (`cuda`), falling back to CPU. Nothing hardcodes a device.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

## 1. Setup

`Settings` is the one object that controls a run. Everything the notebook used
to reassign as a global — the stub confidence, tracing, which agents fabricate,
which router runs — is a field on it, and it travels inside the graph state.
Running a cell twice can therefore no longer leave a knob changed.

In [ ]:
from cricketmind import (CLIP, QUERY, Settings, SHOT_CLASSES, CONF_THRESHOLD,
                         MAX_RETRIES, MAX_REGENS, MAX_ACTIVE_AGENTS,
                         AGENT_REGISTRY, AGENTS_BY_ID, agent_name,
                         build_cricketmind_graph, new_state, describe_state,
                         sanctioned_facts, show_graph)
from cricketmind import steps

LOUD = Settings(trace=True)        # print each node's own log line

print(f"classes {len(SHOT_CLASSES)}   agents {len(AGENT_REGISTRY)}")
print(f"gates   conf>={CONF_THRESHOLD}  retries<={MAX_RETRIES}  "
      f"regens<={MAX_REGENS}  active<={MAX_ACTIVE_AGENTS}")

### Device

Resolved by `src/utils/device.py`, the same helper the training code uses:
XPU → CUDA → CPU, with the right autocast setting for each.

In [ ]:
from src.utils.device import get_amp_settings, get_device

DEVICE = get_device()
AMP_DTYPE, USE_SCALER = get_amp_settings(DEVICE)
print(f"device {DEVICE} | autocast {AMP_DTYPE} | gradscaler {USE_SCALER}")

### The state

Every node reads from and writes into one `TypedDict`, returning only the keys
it changed. How a key merges depends on its reducer:

| Kind | Merge | Used for |
|---|---|---|
| plain key | last write wins | scalars one node writes |
| `Annotated[list, operator.add]` | concatenate | parallel branches |
| `Annotated[list, upsert_findings]` | replace by agent id | findings, which re-run |

In [ ]:
describe_state(new_state(CLIP, QUERY), "INITIAL")

## 2. The vision path — a linear chain

`add_node(name, fn)` registers a node; `add_edge(a, b)` says b runs after a.
A node is registered under the same name as the function that implements it, so
a box in the picture can be found by searching the package for its label.

`compile()` validates the graph — every node reachable, no dangling edges.

In [ ]:
vision = steps.build_vision_graph()
show_graph(vision)

In [ ]:
out = vision.invoke(new_state(CLIP, QUERY, LOUD))
print()
print("label", out["shot_label"], "| p", out["shot_confidence"])
print("shapes", out["frame_features"].shape, "->", out["temporal_features"].shape)

## 3. The Conf. Check and the retry loop

The first place the graph makes a *decision*. A **conditional edge** calls a
function that returns the name of the next route.

Two things that are easy to get wrong:

**A conditional function cannot write to state.** It receives the state and
returns a route name — that is all. So the retry branch cannot point straight
back at N1, because *something* has to increment the counter that makes the loop
terminate. That is what the `retry` node is for.

**The counter is not optional.** LangGraph's `recursion_limit` would eventually
stop a runaway graph, but by raising `GraphRecursionError` — an accident, not a
decision. The counter makes exhaustion a *designed* outcome.

In [ ]:
gated = steps.build_gated_graph()
show_graph(gated)

In [ ]:
cases = [
    ("A: confident first time",     0.92,               "ok",             0),
    ("B: recovers on third attempt", (0.41, 0.55, 0.88), "ok",             2),
    ("C: never recovers, degrades",  0.30,              "low_confidence", MAX_RETRIES),
]
for title, confidence, want_status, want_retries in cases:
    out = gated.invoke(new_state(CLIP, QUERY, Settings(confidence=confidence)))
    print(f"{title:32s} -> status={out['status']:16s} retries={out['retry_count']}")
    assert out["status"] == want_status and out["retry_count"] == want_retries
print()
print("PASS: the gate accepts, retries with a bound, and degrades instead of looping")

## 4. The text branch and the Shot DNA join

### An edge is a trigger, not a data dependency

This is the trap in this stage, and it cost a broken run to find.

Figure 1 draws an arrow from N3 into N6, so the obvious wiring is
`add_edge("n3_text_encoder", "n6_shot_dna")`. That is wrong: N6 fires the moment
N3 finishes, while the vision branch is still at N2.

| wiring | accept path | degrade path |
|---|---|---|
| static `n3 → n6` edge | N6 fires early, crashes | crashes |
| two static parents | still fires early | still fires early |
| two static parents, `defer=True` | works | **N6 runs anyway** |
| accept route only | works | N6 correctly skipped |

**In LangGraph an edge means "run this next", not "this node needs that node's
output".** A node fires as soon as *any* incoming edge is satisfied — there is no
all-parents barrier. The data dependency is real, but it travels through
**state**, not through an edge.

In [ ]:
grounded = steps.build_grounded_graph()
show_graph(grounded)

In [ ]:
# accept: both branches ran, N6 joined them
a = grounded.invoke(new_state(CLIP, QUERY))
assert a["shot_dna"]["shot"] == "Cover Drive"
assert a["shot_dna"]["filters"]["player"] == "Kohli"

# degrade: N3 still ran, but N6 was skipped rather than run half-fed
c = grounded.invoke(new_state(CLIP, QUERY, Settings(confidence=0.30)))
assert c["status"] == "low_confidence"
assert "query_embedding" in c, "N3 is independent of the gate"
assert c.get("shot_dna") is None, "N6 must not run when the gate refused"
print("OK: N6 joins on accept, and is skipped — not deadlocked — on degrade")

## 5. Retrieval — and why the reducers exist

Two edges out of N6 means both retrievers run in the same superstep. They write
different keys, so both writes must survive: that is what
`Annotated[list, operator.add]` is for.

The cell below shows what happens *without* a reducer — two nodes writing one
plain key in one superstep.

In [ ]:
retrieval = steps.build_retrieval_graph()
out = retrieval.invoke(new_state(CLIP, QUERY))
print("stats_hits  ", len(out["stats_hits"]), "deliveries")
print("context_hits", len(out["context_hits"]), "rows")
assert len(out["stats_hits"]) == 100 and len(out["context_hits"]) == 4
print("OK: parallel writes merged, nothing dropped")

In [ ]:
from langgraph.graph import END, START, StateGraph
from cricketmind.state import CricketMindState

def _writer(tag):
    def _w(state):
        return {"report": f"from {tag}"}      # `report` has NO reducer
    return _w

_g = StateGraph(CricketMindState)
_g.add_node("a", _writer("A")); _g.add_node("b", _writer("B"))
_g.add_edge(START, "a"); _g.add_edge(START, "b")
_g.add_edge("a", END);   _g.add_edge("b", END)
try:
    _g.compile().invoke(new_state("c.mp4", "q"))
    print("no error -- this field must have a reducer after all")
except Exception as e:
    print(f"{type(e).__name__}: {str(e).splitlines()[0]}")
    print()
    print("That is what stats_hits and context_hits would do without")
    print("Annotated[list, operator.add]. The reducer is not decoration.")

## 6. The two sanctioned engines

N8 counts; N9 simulates. These are the **only** components permitted to produce
a number, and everything downstream must cite them.

N9's Monte Carlo result is checked against the closed form `1 − (1−p)⁶`, so the
simulator is verified rather than trusted.

In [ ]:
analytics = steps.build_analytics_graph()
out = analytics.invoke(new_state(CLIP, QUERY, LOUD))
print()
for shot, a in out["stats_results"]["by_shot"].items():
    print(f"  {shot:12s} SR {a['strike_rate']:6.1f}  over {a['balls']} balls")

sim = out["sim_results"]
lo, hi = sim["ci95"]
print(f"\n  simulated {sim['p_dismissal']:.1%}  analytic {sim['analytic_check']:.1%}"
      f"  95% CI {lo:.1%}-{hi:.1%}")
assert lo <= sim["analytic_check"] <= hi, "closed form outside the interval"
print("\nOK: SR 142 vs 112 counted from rows; Monte Carlo agrees with the closed form")

## 7. The router — the LLM proposes, the rules decide

Routing has two stages, deliberately kept apart:

1. **propose** — the keyword scorer, or a language model.
2. **decide** — fixed rules, identical for either: agents whose evidence does
   not exist are removed, at most four are kept, and if none remain the default
   pair answers.

Because the rules run *after* the proposal, the cap and the evidence rule hold
whichever router proposed. That is what makes the language model safe to use
here: it can suggest, but it cannot exceed a bound.

In [ ]:
router = steps.build_router_graph()

probes = [
    ("workload / injury", "How much workload has Kohli carried this week, any injury risk?"),
    ("market valuation",  "What is Kohli's IPL auction valuation and market worth?"),
    ("broadcast",         "Generate commentary and a wagon wheel overlay clip"),
    ("compliance",        "Check ICC eligibility and audit the franchise"),
    ("vague",             "Tell me about this delivery"),
]
for label, q in probes:
    res = router.invoke(new_state(CLIP, q))
    print(f"  {label:18s} -> {[agent_name(a) for a in res['active_agents']]}")

In [ ]:
# the cap holds even when every keyword fires
everything = ("budget auction squad tactic bowling technique shot scout league "
              "fan fantasy pitch prep crowd sponsor logo roi commentary overlay "
              "eligibility icc academy drill injury workload valuation market")
res = router.invoke(new_state(CLIP, everything))
print(f"query matching every agent -> woke {len(res['active_agents'])} of {len(AGENT_REGISTRY)}")
assert len(res["active_agents"]) == MAX_ACTIVE_AGENTS
print("OK: dispatch stays bounded")

## 8. The twelve agents — one factory, a list fan-out

The agents differ only in voice and in which facts they may quote, so that
difference lives in data (`AGENT_REGISTRY`) and one factory builds all twelve.

A conditional function returning a **list** is LangGraph's fan-out: every name
it returns runs in the same superstep. Returning `[END]` rather than `[]`
matters — an empty list leaves the graph unable to finish.

In [ ]:
agents = steps.build_agent_graph()
out = agents.invoke(new_state(CLIP, QUERY))
for f in out["agent_findings"]:
    print(f"[{f['stakeholder']}]")
    print(f"  {f['text']}")
    print(f"  cites: {f['cites']}\n")

facts = sanctioned_facts(out)
for f in out["agent_findings"]:
    for k, v in f["cites"].items():
        assert facts.get(k) == v, f"{f['agent']} cited an unsanctioned {k}"
print("OK: only the routed agents ran, and every figure came from N8 or N9")

## 9. The finished graph

All 30 nodes, both correction loops, both entry paths.

`defer=True` goes on nodes whose parents sit at **different depths** (N6, N8,
N9, N10). It does **not** go on `faith_check`: the agents are siblings from one
fan-out, so they finish in the same superstep. Deferring it also made
`get_graph()` draw a phantom `n13 → faith_check` edge and lose `n13 → END` —
the runtime was fine, but the *picture* was wrong, and the picture is what goes
in the thesis.

In [ ]:
cricketmind = build_cricketmind_graph()
nodes = [n for n in cricketmind.get_graph().nodes if not n.startswith("__")]
print(len(nodes), "nodes")
show_graph(cricketmind)

In [ ]:
final = cricketmind.invoke(new_state(CLIP, QUERY, LOUD))
print()
print(final["report"])

### Two ways in

A text-only question bypasses the whole vision path. Posture is left as `None`,
never defaulted — a default would be an invented measurement entering the
sanctioned facts, where the Faith Check would then certify it as ground truth.
A fabrication laundered through the mechanism built to stop fabrications.

The report is shorter instead, because the router declines to wake the two
agents that must cite a posture measurement.

In [ ]:
text_only = cricketmind.invoke(new_state(None, QUERY))
print(text_only["report"])
print()
print("woken:", [agent_name(a) for a in text_only["active_agents"]])
assert text_only["shot_dna"]["posture"] is None, "posture must not be invented"

### The Faith Check catching a fabrication

`Settings.lie_once` makes a named agent corrupt one of the figures it cites on
its first attempt. The gate rejects it, only that agent is re-run, and the
corrected answer reaches the report.

`lie_always` never corrects itself: the budget is spent, the finding is
withheld, and the report says so.

In [ ]:
LIAR = "n11b_tactical_analysis"

fixed = cricketmind.invoke(new_state(CLIP, QUERY, Settings(lie_once={LIAR}, trace=True)))
print(f"\nregens {fixed['regen_count']} | status {fixed['status']} | "
      f"999.0 in report: {'999.0' in fixed['report']}")

withheld = cricketmind.invoke(new_state(CLIP, QUERY, Settings(lie_always={LIAR})))
print(f"regens {withheld['regen_count']} | status {withheld['status']} | "
      f"999.0 in report: {'999.0' in withheld['report']}")
print()
print(withheld["report"])

---

## Where everything lives

| What | Where |
|---|---|
| Constants and `Settings` | `cricketmind/config.py` |
| State, reducers, `trace` | `cricketmind/state.py` |
| Node bodies | `cricketmind/nodes/` |
| The twelve agents, sanctioned facts | `cricketmind/agents.py` |
| The synthetic corpus | `cricketmind/corpus.py` |
| The full graph | `cricketmind/graph.py` |
| These seven step graphs | `cricketmind/steps.py` |
| The language-model router | `cricketmind/llm_router.py` |
| Measurements and paper tables | `cricketmind/experiments/` |
| Checks | `tests/` — run `python -m pytest` |

The measured results are in **`02_results.ipynb`**.